# Week 01 practical baseline

This notebook loads `Qwen/Qwen2.5-0.5B-Instruct`, tests chat generation, measures cold/warm latency, and writes `baseline.json`. On Apple Silicon it uses PyTorch MPS, not CUDA or MLX. Run top to bottom.

In [5]:
import json, platform, statistics, time
from pathlib import Path
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float16 if DEVICE in ('mps', 'cuda') else torch.float32
def synchronize_device():
    if DEVICE == 'cuda': torch.cuda.synchronize()
    elif DEVICE == 'mps': torch.mps.synchronize()

print({'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': DEVICE})

{'python': '3.12.10', 'torch': '2.13.0', 'transformers': '5.16.1', 'device': 'mps'}


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
model.to(DEVICE)
model.eval()
print('model loaded:', MODEL_ID, 'parameters:', sum(p.numel() for p in model.parameters()))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

model loaded: Qwen/Qwen2.5-0.5B-Instruct parameters: 494032768


In [7]:
# Inspect the two representations before the model sees them.
messages = [{'role': 'user', 'content': 'Explain caching in one sentence.'}]
rendered = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
print('Rendered chat text:\n', repr(rendered))
encoded = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt')
print('input_ids shape:', tuple(encoded['input_ids'].shape))
print('input_ids:', encoded['input_ids'][0].tolist())
print('tokens:', tokenizer.convert_ids_to_tokens(encoded['input_ids'][0]))
print('attention_mask:', encoded['attention_mask'][0].tolist())

Rendered chat text:
 '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nExplain caching in one sentence.<|im_end|>\n<|im_start|>assistant\n'
input_ids shape: (1, 36)
input_ids: [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 840, 20772, 47430, 304, 825, 11652, 13, 151645, 198, 151644, 77091, 198]
tokens: ['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'ĠQ', 'wen', ',', 'Ġcreated', 'Ġby', 'ĠAlibaba', 'ĠCloud', '.', 'ĠYou', 'Ġare', 'Ġa', 'Ġhelpful', 'Ġassistant', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'user', 'Ċ', 'Ex', 'plain', 'Ġcaching', 'Ġin', 'Ġone', 'Ġsentence', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'assistant', 'Ċ']
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


`apply_chat_template` first renders the list of role/content messages using the model's template (a Jinja template stored with the tokenizer), adding model-specific control tokens and the assistant-generation marker. With `tokenize=True`, that rendered conversation is immediately converted into token IDs. `attention_mask` marks which positions are real tokens (`1`) versus padding (`0`). This notebook uses one request at a time, so the mask is usually all ones; it becomes essential when padding a batch of different-length prompts.

In [8]:
def make_inputs(text):
    messages = [{'role': 'user', 'content': text}]
    encoded = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt')
    return {key: value.to(DEVICE) for key, value in encoded.items()}

@torch.inference_mode()
def generate_once(text, max_new_tokens=64):
    inputs = make_inputs(text)
    synchronize_device()
    started = time.perf_counter()
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    synchronize_device()
    elapsed = time.perf_counter() - started
    new_tokens = output[0, inputs['input_ids'].shape[-1]:]
    return {'text': tokenizer.decode(new_tokens, skip_special_tokens=True), 'prompt_tokens': int(inputs['input_ids'].shape[-1]), 'output_tokens': int(new_tokens.shape[-1]), 'total_latency_seconds': elapsed, 'output_tokens_per_second': int(new_tokens.shape[-1]) / elapsed if elapsed else None}

input_text = 'Explain caching in one sentence.'
print('input:', make_inputs(input_text))
smoke = generate_once('Explain caching in one sentence.')
print(smoke)

input: {'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,    840,  20772,  47430,
            304,    825,  11652,     13, 151645,    198, 151644,  77091,    198]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}
{'text': 'Caching is the process of storing frequently accessed data in memory to reduce the number of requests made to the server, improving performance and reducing load on the server.', 'prompt_tokens': 36, 'output_tokens': 33, 'total_latency_seconds': 2.7242836250006803, 'output_tokens_per_second': 12.113276201185462}


## Streaming smoke test

This uses a background generation thread so tokens can be consumed as they become available. The first yielded text is the TTFT boundary.

In [9]:
import threading

def stream_once(text, max_new_tokens=64):
    inputs = make_inputs(text)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(inputs, streamer=streamer, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    thread = threading.Thread(target=model.generate, kwargs=kwargs)
    started = time.perf_counter(); thread.start()
    chunks = []; ttft = None
    print('stream: ', end='', flush=True)
    for chunk in streamer:
        if ttft is None: ttft = time.perf_counter() - started
        chunks.append(chunk)
        print(chunk, end='', flush=True)
    thread.join(); synchronize_device(); total = time.perf_counter() - started
    print()
    return {'text': ''.join(chunks), 'prompt_tokens': int(inputs['input_ids'].shape[-1]), 'output_tokens': len(tokenizer.encode(''.join(chunks), add_special_tokens=False)), 'ttft_seconds': ttft, 'total_latency_seconds': total}

print(stream_once('Explain prefill and decode in simple terms.'))

{'text': 'Certainly! Let\'s break down the concepts of "prefill" and "decode" in a way that is easy to understand.\n\n### Prefill\n\n**Prefill** refers to the process of preparing or setting up something before it needs to be used. It involves creating a template or blueprint for what you want to include', 'prompt_tokens': 39, 'output_tokens': 64, 'ttft_seconds': 0.09205254200060153, 'total_latency_seconds': 1.2383265829994343}


In [10]:
prompts = {
    'short': 'Explain caching in one sentence.',
    'medium': 'Explain how an HTTP request becomes generated tokens in 150 words.',
    'long': ('Explain how an HTTP request becomes generated tokens. ' * 120),
}

# Warm up once, then collect five measured runs per prompt.
results = []
for name, prompt in prompts.items():
    generate_once(prompt)
    for repeat in range(5):
        row = generate_once(prompt)
        row.update({'prompt_name': name, 'repeat': repeat + 1, 'model_id': MODEL_ID, 'device': DEVICE})
        results.append(row)

for name in prompts:
    rows = [r for r in results if r['prompt_name'] == name]
    latencies = [r['total_latency_seconds'] for r in rows]
    print(name, {'prompt_tokens': rows[0]['prompt_tokens'], 'latency_median': statistics.median(latencies), 'latency_p95_approx': sorted(latencies)[min(len(latencies)-1, int(len(latencies)*0.95))]})

short {'prompt_tokens': 36, 'latency_median': 0.5491790829983074, 'latency_p95_approx': 0.5620779999990191}
medium {'prompt_tokens': 45, 'latency_median': 1.0661002079978061, 'latency_p95_approx': 1.0674458749999758}
long {'prompt_tokens': 1111, 'latency_median': 1.3583051249988785, 'latency_p95_approx': 1.5008537499998056}


In [12]:
output = {
    'model_id': MODEL_ID,
    'device': DEVICE,
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__,
    'generation': {'max_new_tokens': 64, 'do_sample': False, 'warmups_per_prompt': 1, 'measured_runs_per_prompt': 5},
    'results': results,
}
cwd = Path.cwd()
week1_dir = cwd / 'week-01-baseline-server' if (cwd / 'week-01-baseline-server').is_dir() else cwd
week1_dir.mkdir(parents=True, exist_ok=True)
output_path = week1_dir / 'baseline.json'
output_path.write_text(json.dumps(output, indent=2))
print(f'wrote {output_path.resolve()}')

wrote /Users/myatkaung/Desktop/production-llm-inference-lab/week-01-baseline-server/baseline.json
